<img src="../img/viu_logo.png" width="200">

# Aprendizaje por Refuerzo: Proyecto de programación

- **Profesor:** Francisco Muñoz Montoya
- **Autores:** Guillermo Sanchez Garcia, Diego Aguado Sanchez, Alejandro Pequeno Lizcano, Jaume Montanera

### Índice
0. [Introducción](#0-introducción)
1. [Implementación red neuronal](#1-implementación-red-neuronal)
2. [Implementación del agente (DQN)](#2-implementación-del-agente)
3. [Conclusiones](#3-conclusiones)

## 0. Introducción

Consideraciones a tener en cuenta:

- El entorno sobre el que trabajaremos será el indicado en el listado correspondien de cada grupo y el algoritmo que usaremos será _DQN_.

- Para nuestro ejercicio, el requisito mínimo será alcanzado cuando el agente consiga una **media de recompensa por encima de los puntos indicados en el listado por grupos en modo test**. Por ello, esta media de la recompensa se calculará a partir del código de test en la última celda del notebook.

Este proyecto práctico consta de tres partes:

1.   Implementar la red neuronal que se usará en la solución
2.   Implementar las distintas piezas de la solución DQN y probar al menos 3 propuestas diferentes de mejora.
3.   Justificar la respuesta en relación a los resultados obtenidos e incluir al menos 3 gráficas relevantes comparando las 3 propuestas.

**Rúbrica**: Se valorará la originalidad en la solución aportada, así como la capacidad de discutir los resultados de forma detallada. El requisito mínimo servirá para aprobar la actividad, bajo premisa de que la discusión del resultado sera apropiada.

IMPORTANTE:

* Si no se consigue una puntuación óptima, responder sobre la mejor puntuación obtenida.
* Para entrenamientos largos, recordad que podéis usar checkpoints de vuestros modelos para retomar los entrenamientos. En este caso, recordad cambiar los parámetros adecuadamente (sobre todo los relacionados con el proceso de exploración).
* Se deberá entregar unicamente el notebook y los pesos del mejor modelo en un fichero .zip, de forma organizada.
* Cada alumno deberá de subir la solución de forma individual.

## Imports, configuración y entorno

In [1]:
from __future__ import division

# Librerías estándar
# ===========================================================
import os
import sys
import types
import platform
import random
from pathlib import Path
import numpy as np

# Variables de entorno
# ===========================================================
from dotenv import load_dotenv

load_dotenv()

# keras-rl2 usa tensorflow.keras internamente — debe establecerse antes de importar TF
# ===========================================================
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# Deep learning: TF
# ===========================================================
import tensorflow as tf

# Keras 2 (tf-keras): modelos, capas, optimizadores
# ===========================================================
import tf_keras
from tf_keras.models import Sequential
from tf_keras.layers import Dense, Activation, Flatten, Convolution2D, Permute
from tf_keras.optimizers import Adam
import tf_keras.backend as K

# Shim de compatibilidad: keras-rl2 1.0.5 usa tensorflow.python.keras (eliminado en TF 2.16+)
# ===========================================================
import tensorflow.keras as _tfkeras

if not hasattr(_tfkeras, "__version__"):
    _tfkeras.__version__ = tf_keras.__version__

_compat_generic = types.ModuleType("tensorflow.python.keras.utils.generic_utils")
_compat_generic.Progbar = tf_keras.utils.Progbar

_compat_utils = types.ModuleType("tensorflow.python.keras.utils")
_compat_utils.generic_utils = _compat_generic

_compat_keras = types.ModuleType("tensorflow.python.keras")
_compat_keras.callbacks = tf_keras.callbacks
_compat_keras.utils = _compat_utils

sys.modules.setdefault("tensorflow.python.keras", _compat_keras)
sys.modules.setdefault("tensorflow.python.keras.callbacks", tf_keras.callbacks)
sys.modules.setdefault("tensorflow.python.keras.utils", _compat_utils)
sys.modules.setdefault("tensorflow.python.keras.utils.generic_utils", _compat_generic)

# Entorno RL
# ===========================================================
import gym
from PIL import Image

# keras-rl2: agente DQN, políticas, memoria
# ===========================================================
from rl.agents.dqn import DQNAgent
from rl.policy import LinearAnnealedPolicy, BoltzmannQPolicy, EpsGreedyQPolicy
from rl.memory import SequentialMemory
from rl.core import Processor
from rl.callbacks import FileLogger, ModelIntervalCheckpoint

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
IS_ARM_MAC = sys.platform == "darwin" and platform.machine() == "arm64"

print("TensorFlow version:", tf.__version__)
print("tf-keras version:  ", tf_keras.__version__)
print("ARM Mac:           ", IS_ARM_MAC)
print("GPUs disponibles:  ", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.18.0
tf-keras version:   2.18.0
ARM Mac:            True
GPUs disponibles:   [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Constantes

In [5]:
# Path base
BASE_FOLDER = Path(os.getenv("BASE_FOLDER"))
SEED = int(os.getenv("SEED", 42))
np.random.seed(SEED)

# Adaptar a cada entorno según corresponda
INPUT_SHAPE = (84, 84)
WINDOW_LENGTH = 4

env_name = "SpaceInvaders-v0"  # nombre del entorno
env = gym.make(env_name)
env.seed(SEED)
nb_actions = env.action_space.n
print(f"Entorno: {env_name} | Acciones: {nb_actions}")

Entorno: SpaceInvaders-v0 | Acciones: 6


## 1. Implementación red neuronal

In [6]:
class AtariProcessor(Processor):
    def process_observation(self, observation):
        assert observation.ndim == 3  # (height, width, channel)
        img = Image.fromarray(observation)
        img = img.resize(INPUT_SHAPE).convert("L")
        processed_observation = np.array(img)
        assert processed_observation.shape == INPUT_SHAPE
        return processed_observation.astype("uint8")

    def process_state_batch(self, batch):
        processed_batch = batch.astype("float32") / 255.0
        return processed_batch

    def process_reward(self, reward):
        return np.clip(reward, -1.0, 1.0)

## 2. Implementación del agente (DQN)

## 3. Conclusiones
Justificación de los parámetros seleccionados y de los resultados obtenidos

